# Part 1 · Dataset Exploration & Preparation
**Owner: Teddy**

### Setup (VS Code / local) — run first

In [ ]:
import os, zipfile, pathlib, shutil
# Unzip data/realwaste.zip -> data/RealWaste/<category>/ if not already done
if not pathlib.Path("data/RealWaste").exists() and pathlib.Path("data/realwaste.zip").exists():
    with zipfile.ZipFile("data/realwaste.zip") as z: z.extractall("data/_tmp")
    for p in pathlib.Path("data/_tmp").rglob("RealWaste"):
        shutil.move(str(p),"data/RealWaste"); break
    shutil.rmtree("data/_tmp", ignore_errors=True)
IMAGE_DIR = "data/RealWaste"
print("Images present:", os.path.isdir(IMAGE_DIR))

In [ ]:
import os, re, json, random, pathlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
np.random.seed(42); random.seed(42)
IMG_HEIGHT = IMG_WIDTH = 224
BATCH_SIZE = 32

### 1.1 Image distribution & characteristics

In [ ]:
class_names = sorted([d.name for d in pathlib.Path(IMAGE_DIR).glob("*") if d.is_dir()])
counts = {c: len(list((pathlib.Path(IMAGE_DIR)/c).glob("*"))) for c in class_names}
print("Classes:", class_names)
pd.Series(counts).sort_values().plot.barh(title="Images per category"); plt.tight_layout(); plt.show()
print("Imbalance ratio (max/min):", round(max(counts.values())/min(counts.values()),2))

In [ ]:
from PIL import Image
fig,ax=plt.subplots(3,3,figsize=(9,9))
for a,c in zip(ax.ravel(),class_names):
    f=next((pathlib.Path(IMAGE_DIR)/c).glob("*")); im=Image.open(f)
    a.imshow(im); a.set_title(f"{c}\n{im.size}"); a.axis("off")
plt.tight_layout(); plt.show()

### 1.2 Text descriptions & policy documents

In [ ]:
desc = pd.read_csv("data/waste_descriptions.csv")
print(desc.shape); display(desc.head())
print(desc["category"].value_counts())

In [ ]:
with open("data/waste_policy_documents.json") as f: policies = json.load(f)
print(len(policies),"policy documents")
for p in policies[:3]: print("-", p["policy_type"], p["categories_covered"])

### 1.3 Data pipelines (image splitter + text split)

In [ ]:
import tensorflow as tf
data_dir = pathlib.Path(IMAGE_DIR)
class_names = sorted([i.name for i in data_dir.glob("*") if i.is_dir()])
num_classes = len(class_names); print("Classes:", class_names)
train_ds = tf.keras.utils.image_dataset_from_directory(data_dir, validation_split=0.2,
    subset="training", seed=42, image_size=(IMG_HEIGHT,IMG_WIDTH), batch_size=BATCH_SIZE,
    label_mode="categorical", shuffle=True)
validation_ds = tf.keras.utils.image_dataset_from_directory(data_dir, validation_split=0.2,
    subset="validation", seed=42, image_size=(IMG_HEIGHT,IMG_WIDTH), batch_size=BATCH_SIZE,
    label_mode="categorical", shuffle=True)
val_batches = tf.data.experimental.cardinality(validation_ds)
test_dataset = validation_ds.take(val_batches//2); validation_ds = validation_ds.skip(val_batches//2)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(AUTOTUNE)
validation_ds = validation_ds.cache().prefetch(AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(AUTOTUNE)

In [ ]:
from sklearn.model_selection import train_test_split
def clean_text(t):
    t=str(t).lower(); t=re.sub(r"[^a-z\s]"," ",t); return re.sub(r"\s+"," ",t).strip()
train_txt, hold = train_test_split(desc, test_size=0.3, stratify=desc["category"], random_state=42)
val_txt, test_txt = train_test_split(hold, test_size=0.5, stratify=hold["category"], random_state=42)
print("text train/val/test:", len(train_txt), len(val_txt), len(test_txt))

### TODO (Teddy)\n- Comment on resolution/background/quality challenges\n- Justify the splits; note class imbalance & how models handle it\n- Document dataset biases/limitations